# Job Posting Velocity as a Lead Indicator for Earnings Surprise Direction

**Author**: Research Pipeline  
**Strategy**: Alternative Data — Job Posting Velocity  
**Notebook**: 01 — Exploratory Analysis & Full Pipeline Walkthrough

---

This notebook walks through the full research pipeline from raw Wayback Machine snapshots
to hypothesis testing and backtest construction.  Each section can be run independently
provided the prior sections' cached outputs exist in `data/processed/`.

In [ ]:
import sys
import os

# Add project root to path so src.* imports work from the notebook
sys.path.insert(0, os.path.abspath('..'))

import json
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)  # suppress verbose info logs in notebook

import config
from src import wayback_scraper, earnings_data, features, signal, backtest, visualize

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Imports OK — project root:', config.ROOT_DIR)

---
## 1. Universe Overview

The research universe is a hand-curated sample of 15 large-cap S&P 500 companies
spanning five sectors.  The sample was chosen to provide:

- **Wayback Machine coverage**: large-cap companies whose careers pages have been
  archived consistently since 2018.
- **Sector diversity**: to test whether the signal generalises beyond technology.
- **Headcount range**: from ~25k (NVDA) to ~1.5M (AMZN) to test normalisation.

In a production deployment, this would expand to the full S&P 500 or Russell 1000.

In [ ]:
universe = pd.read_csv(config.COMPANY_UNIVERSE_CSV)
print(f'Universe: {len(universe)} companies across {universe["sector"].nunique()} sectors\n')
display(universe.style.background_gradient(subset=['avg_headcount'], cmap='Blues'))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sector_counts = universe['sector'].value_counts()
colors = sns.color_palette('Set2', len(sector_counts))
ax.barh(sector_counts.index, sector_counts.values, color=colors)
ax.set_xlabel('Number of Companies')
ax.set_title('Universe Composition by Sector')
plt.tight_layout()
plt.show()

---
## 2. Wayback Machine Data Acquisition

We use the Wayback Machine CDX API to retrieve archived snapshots of each company's
careers page.  The CDX API returns metadata (timestamps, HTTP status codes) for all
archived captures; we then fetch the HTML for up to 2 snapshots per month.

**Important**: The cell below makes live HTTP requests to the Wayback Machine.  
On first run this may take several minutes per ticker.  Results are cached to
`data/raw/` and `data/processed/` so subsequent runs are instant.

> **Cost to Wayback Machine**: We use `REQUEST_DELAY_SECONDS = 2.0` between calls
> and collapse to 2 snapshots/month to be respectful of their bandwidth.

In [ ]:
# Demo: run CDX lookup for 2 tickers (MSFT and NVDA) over a short window
demo_tickers = [
    ('MSFT', 'https://careers.microsoft.com/us/en'),
    ('NVDA', 'https://nvidia.wd5.myworkdayjobs.com/NVIDIAExternalCareerSite'),
]

demo_from = '20230101'
demo_to   = '20231231'

for ticker, url in demo_tickers:
    print(f'\n--- {ticker} ---')
    snaps = wayback_scraper.get_wayback_snapshots(url, demo_from, demo_to)
    print(f'Found {len(snaps)} snapshots')
    display(snaps.head(4))

In [ ]:
# For a quick local demo without live requests, we synthesise a small timeseries.
# Comment this cell out and uncomment the live-fetch cell above for real data.

import numpy as np
from datetime import datetime

def _synthetic_timeseries(ticker: str, n_months: int = 24, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic monthly posting data for demonstration."""
    rng = np.random.default_rng(seed)
    base = 300 + rng.integers(0, 200)
    counts = base + np.cumsum(rng.normal(5, 20, n_months)).astype(int)
    counts = np.clip(counts, 50, 2000)
    months = pd.date_range('2022-01', periods=n_months, freq='MS')
    month_strs = months.strftime('%Y%m').tolist()
    titles_sample = ['software engineer', 'data scientist', 'product manager',
                     'legal counsel', 'hr generalist', 'sales representative',
                     'backend engineer', 'compliance officer']
    rows = []
    for ym, cnt in zip(month_strs, counts):
        n_titles = min(cnt, 40)
        chosen = rng.choice(titles_sample, size=n_titles, replace=True).tolist()
        rows.append({
            'ticker': ticker,
            'year_month': ym,
            'posting_count': float(cnt),
            'job_titles_raw': json.dumps(chosen),
            'extraction_confidence': rng.choice([0.5, 1.0]),
        })
    return pd.DataFrame(rows)

synthetic_frames = []
for i, row in enumerate(universe.itertuples()):
    synthetic_frames.append(_synthetic_timeseries(row.ticker, n_months=24, seed=i))

raw_timeseries = pd.concat(synthetic_frames, ignore_index=True)
print(f'Synthetic timeseries: {len(raw_timeseries)} rows, {raw_timeseries["ticker"].nunique()} tickers')
raw_timeseries.head(6)

---
## 3. Functional Decomposition

Each job title string is classified into one of seven functional categories
using the keyword taxonomy in `data/reference/job_function_taxonomy.csv`.

The key derived metric is the **signal ratio**:

$$
\text{signal\_ratio} = \frac{\text{engineering} + \text{product} + \text{sales}}{\text{total\_postings}}
$$

A high signal ratio indicates that the company is allocating incremental headcount
to revenue-generating functions, which we interpret as a forward-looking growth signal.

In [ ]:
# Demo: classify a sample of job titles
sample_titles = [
    'Senior Software Engineer',
    'Data Scientist III',
    'Product Manager, Growth',
    'Enterprise Account Executive',
    'Associate General Counsel',
    'HR Business Partner',
    'Controller, North America',
    'Infrastructure Engineer',
    'Supply Chain Analyst',
    'Machine Learning Engineer',
]

cats = features.classify_job_titles(sample_titles)
print('Category breakdown for sample titles:')
for cat, cnt in sorted(cats.items(), key=lambda x: -x[1]):
    if cnt > 0:
        print(f'  {cat:<15}: {cnt}')

# Signal ratio
bullish = cats['ENGINEERING'] + cats['PRODUCT'] + cats['SALES']
total = sum(cats.values())
print(f'\nSignal ratio: {bullish}/{total} = {bullish/total:.2f}')

In [ ]:
# Apply to full synthetic timeseries
enriched = features.compute_functional_decomposition(raw_timeseries)
print(f'Shape after decomposition: {enriched.shape}')
enriched[['ticker','year_month','posting_count','engineering_count',
          'legal_count','signal_ratio']].head(8)

---
## 4. Feature Engineering

We build a feature matrix with four main transformations:

1. **Net-new postings**: month-over-month change (captures velocity, not level)
2. **Headcount normalisation**: divides by company size for cross-sectional comparability
3. **Rolling z-score**: compares current acceleration to the company's own 12-month history
4. **Composite signal**: combines z-score and signal_ratio into a single binary long/short flag

In [ ]:
# Step 1: Net-new postings
enriched = features.compute_net_new_postings(enriched)

# Step 2: Headcount normalisation
enriched = features.normalize_by_headcount(enriched, universe)

# Step 3: Z-score
enriched = features.compute_hiring_acceleration_zscore(enriched)

# Step 4: Composite signal
enriched = features.build_composite_signal(enriched)

print(f'Feature matrix: {enriched.shape}')
print(f'Long signals: {enriched["long_signal"].sum()}')
print(f'Short signals: {enriched["short_signal"].sum()}')
enriched[['ticker','year_month','posting_count','net_new','normalized_net_new',
          'hiring_accel_zscore','signal_ratio','long_signal']].tail(10)

---
## 5. Signal Construction

The joint distribution of `hiring_accel_zscore` and `signal_ratio` defines the
signal space.  The long-signal region is the upper-right quadrant (high z-score,
high signal_ratio), bounded by the threshold lines.

In [ ]:
plot_df = enriched.dropna(subset=['hiring_accel_zscore','signal_ratio'])

fig, ax = plt.subplots(figsize=(9, 6))

long_mask = plot_df['long_signal'] == True
short_mask = plot_df['short_signal'] == True
other_mask = ~long_mask & ~short_mask

ax.scatter(plot_df.loc[other_mask,'hiring_accel_zscore'],
           plot_df.loc[other_mask,'signal_ratio'],
           c='#aaaaaa', alpha=0.4, s=30, label='No Signal')
ax.scatter(plot_df.loc[long_mask,'hiring_accel_zscore'],
           plot_df.loc[long_mask,'signal_ratio'],
           c='#2C7BB6', alpha=0.85, s=60, label='Long Signal', zorder=3)
ax.scatter(plot_df.loc[short_mask,'hiring_accel_zscore'],
           plot_df.loc[short_mask,'signal_ratio'],
           c='#D7191C', alpha=0.85, s=60, label='Short Signal', zorder=3)

ax.axvline(config.HIRING_ACCEL_ZSCORE_THRESHOLD, color='#2C7BB6',
           linestyle='--', linewidth=1.2, alpha=0.7)
ax.axvline(-config.HIRING_ACCEL_ZSCORE_THRESHOLD, color='#D7191C',
           linestyle='--', linewidth=1.2, alpha=0.7)
ax.axhline(config.SIGNAL_RATIO_THRESHOLD, color='#2C7BB6',
           linestyle=':', linewidth=1.2, alpha=0.7)
ax.axhline(0.35, color='#D7191C', linestyle=':', linewidth=1.2, alpha=0.7)

ax.set_xlabel('Hiring Acceleration Z-Score', fontsize=11)
ax.set_ylabel('Signal Ratio (Bullish Functions / Total)', fontsize=11)
ax.set_title('Signal Space: Z-Score × Signal Ratio\nShaded regions = active signals',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f'Long signal region: z > {config.HIRING_ACCEL_ZSCORE_THRESHOLD}, '
      f'ratio > {config.SIGNAL_RATIO_THRESHOLD}')

---
## 6. Hypothesis Testing

**Core hypothesis:**

- H₀: Companies in the long-signal group have the *same* mean EPS surprise as control group
- H₁: Companies in the long-signal group have *higher* mean EPS surprise than control

We test this using a Welch t-test (unequal variances) and a proportions z-test on beat rates.

> ⚠️ With the synthetic data in this notebook, the results are simulated.
> Run with live Wayback + yfinance data for real inference.

In [ ]:
# Synthesise earnings data matched to our universe
rng = np.random.default_rng(0)

earnings_frames = []
for row in universe.itertuples():
    n = 20  # ~5 years of quarterly earnings
    dates = pd.date_range('2019-01-15', periods=n, freq='QS')
    eps_est = rng.uniform(0.5, 5.0, n)
    surprise = rng.normal(0.02, 0.08, n)  # mean 2% beat, 8% std
    eps_act = eps_est * (1 + surprise)
    surprise_pct = surprise * 100
    earnings_frames.append(pd.DataFrame({
        'ticker': row.ticker,
        'earnings_date': dates,
        'eps_estimate': eps_est,
        'eps_actual': eps_act,
        'surprise_pct': surprise_pct,
        'beat_flag': eps_act > eps_est,
    }))

synthetic_earnings = pd.concat(earnings_frames, ignore_index=True)
print(f'Synthetic earnings: {len(synthetic_earnings)} observations')
synthetic_earnings.head(4)

In [ ]:
# Align signal to forward earnings (1Q ahead)
aligned = signal.align_signal_to_earnings(enriched, synthetic_earnings, lead_quarters=1)
print(f'Aligned: {len(aligned)} signal-earnings pairs')
aligned.head(5)

In [ ]:
# Run primary hypothesis test
test_results = signal.run_ttest_validation(aligned)
print('\nNote: With synthetic (random) data, the p-value will not be significant.')
print('This cell demonstrates the output format. Rerun with live data for real inference.')

In [ ]:
# Information Coefficient
ic = signal.compute_information_coefficient(aligned)
print(f'\nIC = {ic:.4f} (synthetic data — not meaningful for inference)')

---
## 7. OLS Regression

We regress forward earnings surprise on the signal features to decompose the
predictive contribution of (a) hiring velocity and (b) functional composition.

HC3-robust standard errors are used to account for heteroskedasticity.

In [ ]:
# Merge sector from universe for sector dummies
aligned_with_sector = aligned.merge(universe[['ticker','sector']], on='ticker', how='left')

try:
    ols_model = signal.run_ols_regression(aligned_with_sector)
    print('\nKey coefficients:')
    print(ols_model.params[['hiring_accel_zscore','signal_ratio']])
except Exception as e:
    print(f'OLS failed (expected with small synthetic sample): {e}')

---
## 8. Backtest Results

We construct an equal-weighted long-short portfolio each month:
- **Long**: top tercile by `signal_strength`
- **Short**: bottom tercile by `signal_strength`

The return proxy is the forward earnings surprise percentage (a proxy for the
abnormal return at the earnings announcement).

In [ ]:
# Run L-S backtest
bt_results = backtest.run_long_short_backtest(aligned_with_sector)
if not bt_results.empty:
    print(f'Backtest: {len(bt_results)} monthly periods')
    display(bt_results.tail(6))
else:
    print('Insufficient signal events for backtest (need ≥3 months with signal)')

In [ ]:
# Performance metrics
if not bt_results.empty:
    metrics = backtest.compute_performance_metrics(bt_results, aligned_with_sector)

In [ ]:
# Visualisations
# Note: figures are saved to outputs/ and displayed inline

# Posting timeseries for one company
fig1 = visualize.plot_posting_timeseries(enriched, 'MSFT')
plt.show()

# Functional decomposition heatmap
fig2 = visualize.plot_functional_decomposition_heatmap(enriched)
plt.show()

# Z-score distribution
fig3 = visualize.plot_zscore_distribution(enriched)
plt.show()

# Signal vs earnings surprise scatter
fig4 = visualize.plot_signal_vs_earnings_surprise(aligned, ic_value=ic)
plt.show()

if not bt_results.empty:
    # Cumulative PnL
    fig5 = visualize.plot_cumulative_pnl(bt_results)
    plt.show()

# Beat rate by quintile (key validation chart)
fig6 = visualize.plot_beat_rate_by_signal_quintile(aligned)
plt.show()

---
## 9. Macro Overlay Analysis

We test whether strategy performance is correlated with the JOLTS Job Openings Rate,
the primary macro regime indicator.

**Hypothesis**: The signal should be more informative in a *slack* labour market
(low JOLTS rate) because company-specific hiring acceleration stands out from the
cross-sectional distribution.  In a universally tight labour market (high JOLTS rate),
all companies are hiring aggressively and the signal is less differentiating.

In [ ]:
# Load JOLTS macro (will hit BLS API on first run, cached thereafter)
try:
    jolts = earnings_data.fetch_jolts_macro(start_year=2019)
    print(f'JOLTS data: {len(jolts)} monthly observations')
    jolts.tail(6)
except Exception as e:
    print(f'JOLTS fetch failed: {e}')
    jolts = pd.DataFrame(columns=['date','jolts_openings_rate'])

In [ ]:
if not jolts.empty and not bt_results.empty:
    # Merge JOLTS into backtest results
    jolts['month'] = pd.to_datetime(jolts['date']).dt.strftime('%Y%m')
    merged = bt_results.merge(jolts[['month','jolts_openings_rate']], on='month', how='left')

    if merged['jolts_openings_rate'].notna().sum() > 5:
        corr = merged['ls_return'].corr(merged['jolts_openings_rate'])
        print(f'Correlation of L-S monthly return with JOLTS openings rate: {corr:.3f}')

        fig, ax = plt.subplots(figsize=(8,5))
        ax.scatter(merged['jolts_openings_rate'], merged['ls_return']*100,
                   alpha=0.7, c='#2C7BB6', edgecolors='white', s=50)
        ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
        ax.set_xlabel('JOLTS Job Openings Rate (%)', fontsize=11)
        ax.set_ylabel('L-S Monthly Return (%)', fontsize=11)
        ax.set_title('Strategy Returns vs. JOLTS Macro Regime', fontsize=12, fontweight='bold')
        ax.text(0.05, 0.95, f'ρ = {corr:.3f}', transform=ax.transAxes,
                fontsize=11, va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        plt.tight_layout()
        plt.show()
    else:
        print('Not enough JOLTS/backtest overlap for regime analysis.')
else:
    print('JOLTS or backtest data not available — skipping regime chart.')

---
## 10. Limitations & Next Steps

### Data Quality Limitations

1. **Wayback Machine coverage gaps**: Not all careers pages are archived consistently.
   Coverage rates vary from ~80% of months for large-cap tech to <30% for mid-cap
   industrials.  Gaps create survivorship bias in the monthly time-series.
   *Mitigation*: track `extraction_confidence` and drop months with confidence < 0.5.

2. **HTML parsing failure rates**: Modern careers pages rely heavily on JavaScript
   rendering; the Wayback Machine may store the pre-render DOM.  Analyse the
   distribution of `extraction_confidence` values — if > 30% of snapshots return
   `confidence = 0.0`, consider integrating a headless-browser scraping step.

3. **yfinance EPS data quality**: yfinance pulls from Yahoo Finance which has
   known data quality issues for pre-2015 estimates and for companies with
   irregular fiscal year ends.  Validate against a paid source (IBES/Refinitiv)
   before using in production.

4. **Survivorship bias in company universe**: The 15-company universe consists of
   companies that are *currently* large-cap S&P 500 members.  A backtest that
   includes only current index constituents will overstate performance because
   it implicitly excludes firms that were delisted or shrank out of the index.

### Scaling Limitations

5. **Scaling to 500+ companies**: At 2 HTTP requests per company per month, a
   500-company universe over 5 years requires ~60,000 Wayback Machine requests.
   At 2s delay per request this would take ~33 hours to build from scratch.
   *Mitigation*: use async requests with a polite concurrency limit (5–10 threads),
   implement delta-scraping (only fetch months not yet cached).

### Next Steps for Production Deployment

- Replace yfinance with IBES/Refinitiv EPS consensus for higher data quality
- Add SEC EDGAR 10-K/10-Q headcount disclosures for more accurate normalisation
- Integrate LinkedIn Insights or Revelio Labs API as a higher-fidelity job data source
- Implement proper walk-forward out-of-sample validation (avoid look-ahead bias)
- Add transaction cost model (market impact, borrow cost for short book)
- Test factor neutralisation (beta, size, momentum, value) to isolate alpha